# Q3: Spelling Verification of ~177,000 Unique Hindi Words

Classifies each word as **correct/incorrect** with confidence scoring.
Includes manual review of low-confidence bucket & failure analysis.

## Setup & Imports

In [1]:
import os
import sys
import pandas as pd
import json
import re
from collections import Counter

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, PROJECT_ROOT)

from src.spelling_checker import HindiSpellingChecker

print(f"Project root: {PROJECT_ROOT}")

Project root: c:\Users\rajee\OneDrive\Desktop\JoshTech tasks


## STEP 1: Load & Explore Data

In [2]:
print("=" * 60)
print("Q3: Spelling Verification")
print("=" * 60)

CSV_PATH = os.path.join(PROJECT_ROOT, "Unique Words Data - Sheet1.csv")
df = pd.read_csv(CSV_PATH)

print(f"Total unique words: {len(df)}")
print(f"Column: {df.columns.tolist()}")
print(f"\nFirst 20 words: {df['word'].head(20).tolist()}")

# Quick data exploration
word_lengths = df['word'].astype(str).str.len()
print(f"\nWord length stats:")
print(f"  Mean: {word_lengths.mean():.1f}")
print(f"  Min:  {word_lengths.min()}")
print(f"  Max:  {word_lengths.max()}")
print(f"  Words with len=1: {(word_lengths == 1).sum()}")
print(f"  Words with len>20: {(word_lengths > 20).sum()}")

Q3: Spelling Verification
Total unique words: 177508
Column: ['word']

First 20 words: ['है', 'तो', 'में', 'जी', 'हैं', 'भी', 'के', 'नहीं', 'कि', 'वो', 'और', 'से', 'जो', 'हो', 'मतलब', 'हां', 'हम', 'की', 'एक', 'ही']

Word length stats:
  Mean: 6.2
  Min:  1
  Max:  233
  Words with len=1: 101
  Words with len>20: 46


## STEP 2: Run Multi-Layer Spelling Checker

In [3]:
print("=" * 60)
print("STEP 2: Run Spelling Checker")
print("=" * 60)

checker = HindiSpellingChecker()

OUTPUT_CSV = os.path.join(PROJECT_ROOT, "results", "q3_spelling_results.csv")
os.makedirs(os.path.dirname(OUTPUT_CSV), exist_ok=True)

stats = checker.check_csv(CSV_PATH, OUTPUT_CSV, word_column='word')

print(f"\n=== RESULTS ===")
print(f"Total words:       {stats['total_words']}")
print(f"Correct spelling:  {stats['correct_spelling']}")
print(f"Incorrect spelling:{stats['incorrect_spelling']}")
print(f"\nConfidence breakdown: {json.dumps(stats['confidence_breakdown'], indent=2)}")
print(f"\nDetection layer breakdown: {json.dumps(stats['layer_breakdown'], indent=2)}")

INFO:src.spelling_checker:Checking 177508 words...


STEP 2: Run Spelling Checker


INFO:src.spelling_checker:Results saved to c:\Users\rajee\OneDrive\Desktop\JoshTech tasks\results\q3_spelling_results.csv
INFO:src.spelling_checker:Stats: {
  "total_words": 177508,
  "correct_spelling": 6692,
  "incorrect_spelling": 170816,
  "confidence_breakdown": {
    "high": 4918,
    "low": 170740,
    "medium": 1850
  },
  "layer_breakdown": {
    "dictionary": 4015,
    "unknown": 170740,
    "single_char": 209,
    "english_transliteration": 664,
    "punctuation": 40,
    "morphology": 1641,
    "number": 123,
    "structure_validation": 76
  }
}



=== RESULTS ===
Total words:       177508
Correct spelling:  6692
Incorrect spelling:170816

Confidence breakdown: {
  "high": 4918,
  "low": 170740,
  "medium": 1850
}

Detection layer breakdown: {
  "dictionary": 4015,
  "unknown": 170740,
  "single_char": 209,
  "english_transliteration": 664,
  "punctuation": 40,
  "morphology": 1641,
  "number": 123,
  "structure_validation": 76
}


## STEP 3: Low-Confidence Bucket Review

In [4]:
print("=" * 60)
print("STEP 3: Low-Confidence Bucket Review")
print("=" * 60)

results_df = pd.read_csv(OUTPUT_CSV)

# Get low-confidence words
low_conf = results_df[results_df['confidence'] == 'low'].copy()
print(f"Low-confidence words: {len(low_conf)}")

# Sample 50 for manual review
sample_size = min(50, len(low_conf))
if sample_size > 0:
    review_sample = low_conf.sample(n=sample_size, random_state=42)
    
    print(f"\nSampled {sample_size} words for manual review:")
    print("-" * 80)
    print(f"{'Word':<20} {'Classification':<15} {'Reason':<45}")
    print("-" * 80)
    
    for _, row in review_sample.iterrows():
        word = str(row['word'])[:18]
        classification = row['classification']
        reason = str(row['reason'])[:43]
        print(f"{word:<20} {classification:<15} {reason:<45}")
    
    # Save review sample
    review_path = os.path.join(PROJECT_ROOT, "results", "q3_low_confidence_review.csv")
    review_sample.to_csv(review_path, index=False, encoding='utf-8-sig')
    print(f"\n✓ Review sample saved to: {review_path}")

STEP 3: Low-Confidence Bucket Review
Low-confidence words: 170740

Sampled 50 words for manual review:
--------------------------------------------------------------------------------
Word                 Classification  Reason                                       
--------------------------------------------------------------------------------
खूमो                 incorrect       Not found in any dictionary or pattern — po  
सुगर,                incorrect       Not found in any dictionary or pattern — po  
डीएस                 incorrect       Not found in any dictionary or pattern — po  
मज़े                 incorrect       Not found in any dictionary or pattern — po  
आजकाल।               incorrect       Not found in any dictionary or pattern — po  
सारेगामापा           incorrect       Not found in any dictionary or pattern — po  
रिक्मेडेशन           incorrect       Not found in any dictionary or pattern — po  
चार.चारचांद          incorrect       Not found in any dictionary or pat

### Manual Review Instructions

For each of the 50 sampled words, verify:
1. Is the word correctly spelled in Hindi/Devanagari?
2. Is it a valid dialect/colloquial/slang word?
3. Is it a transliterated English word?
4. Is it a proper noun (name, place, etc.)?

Add columns:
- `manual_verdict`: correct/incorrect
- `category`: standard_hindi, dialect, colloquial, english_loan, proper_noun, misspelling, unknown

After review, compute: `accuracy = correct_verdicts / total_reviewed × 100%`

## STEP 4: Unreliable Word Categories Analysis

In [5]:
print("=" * 60)
print("STEP 4: Unreliable Word Categories Analysis")
print("=" * 60)

# Analyze patterns in low-confidence words
low_conf_words = low_conf['word'].astype(str).tolist()

# Category 1: Potential dialect/regional words (common patterns)
dialect_patterns = [w for w in low_conf_words if any(w.endswith(s) for s in ['िये', 'ियो', 'एंगे', 'ाओ'])]

# Category 2: Compound words (long, no spaces)
long_words = [w for w in low_conf_words if len(w) > 15]

# Category 3: Words with nukta (ज़, फ़, etc. — Urdu influence)
nukta_words = [w for w in low_conf_words if '़' in w]

print(f"""
=== UNRELIABLE WORD CATEGORIES ===

CATEGORY 1: Regional Dialect Words
  - Words from regional Hindi dialects (Bhojpuri, Rajasthani, Awadhi, etc.)
  - These are valid spoken Hindi but absent from standard dictionaries
  - Count in low-confidence: {len(dialect_patterns)} with dialect-like suffixes
  - WHY UNRELIABLE: Standard Hindi dictionaries don't cover dialect vocabulary.
    These words may be perfectly valid in their regional context but will always
    be flagged as unknown by dictionary-based approaches.

CATEGORY 2: Borrowed/Nukta Words (Urdu-Hindi overlap)
  - Words using nukta (़) for sounds borrowed from Urdu/Arabic/Persian
  - Examples: ज़रूर vs जरूर, फ़ोन vs फोन
  - Count: {len(nukta_words)} words with nukta characters
  - WHY UNRELIABLE: Alternate spellings with/without nukta are both valid.
    The system may classify one form as correct and the other as misspelled,
    when both are acceptable in modern Hindi usage.
""")

STEP 4: Unreliable Word Categories Analysis

=== UNRELIABLE WORD CATEGORIES ===

CATEGORY 1: Regional Dialect Words
  - Words from regional Hindi dialects (Bhojpuri, Rajasthani, Awadhi, etc.)
  - These are valid spoken Hindi but absent from standard dictionaries
  - Count in low-confidence: 995 with dialect-like suffixes
  - WHY UNRELIABLE: Standard Hindi dictionaries don't cover dialect vocabulary.
    These words may be perfectly valid in their regional context but will always
    be flagged as unknown by dictionary-based approaches.

CATEGORY 2: Borrowed/Nukta Words (Urdu-Hindi overlap)
  - Words using nukta (़) for sounds borrowed from Urdu/Arabic/Persian
  - Examples: ज़रूर vs जरूर, फ़ोन vs फोन
  - Count: 7926 words with nukta characters
  - WHY UNRELIABLE: Alternate spellings with/without nukta are both valid.
    The system may classify one form as correct and the other as misspelled,
    when both are acceptable in modern Hindi usage.



## STEP 5: Final Summary

In [6]:
print("=" * 60)
print("STEP 5: Deliverables Summary")
print("=" * 60)

print(f"""
DELIVERABLES:
  ✓ Total unique words analyzed: {stats['total_words']}
  ✓ Correctly spelled words: {stats['correct_spelling']}
  ✓ Incorrectly spelled words: {stats['incorrect_spelling']}
  
  Output files:
    1. {OUTPUT_CSV} — Full word list with classification
    2. {os.path.join(PROJECT_ROOT, 'results', 'q3_low_confidence_review.csv')} — Low-confidence review sample
    
  Next steps:
    - Upload results CSV to Google Sheets
    - Complete manual review of 50 low-confidence words
    - Document accuracy analysis
""")

print("\n✓ Q3 Complete!")

STEP 5: Deliverables Summary

DELIVERABLES:
  ✓ Total unique words analyzed: 177508
  ✓ Correctly spelled words: 6692
  ✓ Incorrectly spelled words: 170816
  
  Output files:
    1. c:\Users\rajee\OneDrive\Desktop\JoshTech tasks\results\q3_spelling_results.csv — Full word list with classification
    2. c:\Users\rajee\OneDrive\Desktop\JoshTech tasks\results\q3_low_confidence_review.csv — Low-confidence review sample
    
  Next steps:
    - Upload results CSV to Google Sheets
    - Complete manual review of 50 low-confidence words
    - Document accuracy analysis


✓ Q3 Complete!
